In [46]:
import pandas as pd
import numpy as np

In [70]:
brenda = pd.read_csv('data/brenda_data_preprocessed.csv')
sabiork = pd.read_csv('data/sabiork_data_preprocessed.csv')

In [72]:
brenda.columns

Index(['substrate', 'organism', 'pH', 'temperature', 'type', 'sequence',
       'turnover_number', 'enzyme_name', 'ec_number', 'uniprot', 'commentary',
       'mutation', 'isomeric_smiles', 'canonical_smiles'],
      dtype='object')

In [73]:
sabiork.columns

Index(['substrate', 'organism', 'pH', 'temperature', 'type', 'sequence',
       'turnover_number', 'Enzyme Variant', 'ec_number', 'uniprot',
       'parameter.name', 'parameter.type', 'parameter.associatedSpecies',
       'parameter.startValue', 'parameter.endValue',
       'parameter.standardDeviation', 'parameter.unit', 'mutation',
       'isomeric_smiles', 'canonical_smiles'],
      dtype='object')

In [74]:
sabiork.isna().sum()

substrate                          0
organism                           0
pH                                 0
temperature                        0
type                               0
sequence                           0
turnover_number                    0
Enzyme Variant                     0
ec_number                          0
uniprot                            0
parameter.name                     0
parameter.type                     0
parameter.associatedSpecies    14925
parameter.startValue               0
parameter.endValue             14861
parameter.standardDeviation        0
parameter.unit                     0
mutation                        9287
isomeric_smiles                11439
canonical_smiles               11439
dtype: int64

In [75]:
sabiork['pH'] = pd.to_numeric(sabiork['pH'], errors='coerce')
sabiork['temperature'] = pd.to_numeric(sabiork['temperature'], errors='coerce')

In [76]:
common_columns = ['substrate', 'organism', 'pH', 'temperature', 'type', 'sequence',
                  'turnover_number', 'ec_number', 'uniprot', 'mutation', 
                  'isomeric_smiles', 'canonical_smiles']

In [77]:
brenda_df = brenda[common_columns]
sabiork_df = sabiork[common_columns]

combined = pd.concat([brenda_df, sabiork_df], axis=0)

In [78]:
combined = combined.dropna(subset=['pH', 'temperature', 'isomeric_smiles', 'canonical_smiles'])

In [79]:
len(combined)

14730

In [ ]:
group_columns = ['substrate', 'organism', 'pH', 'temperature', 'type', 'sequence']

combined['turnover_number'] = pd.to_numeric(combined['turnover_number'], errors='coerce')

df_clean = combined.dropna(subset=['turnover_number'])

df_aggregated = df_clean.groupby(group_columns).agg({
    'turnover_number': lambda x: np.exp(np.mean(np.log(x[x > 0]))) if (x > 0).any() else np.nan
}).reset_index()

for col in df_clean.columns:
    if col not in group_columns and col != 'turnover_number':
        df_aggregated[col] = df_clean.groupby(group_columns)[col].first().values

print("Original data length:", len(combined))
print("After refining:", len(df_clean))
print("After aggregation:", len(df_aggregated))
print("Columns:", df_aggregated.columns.tolist())

원본 데이터 개수: 14730
정제 후 데이터 개수: 14730
집계 후 데이터 개수: 14718
컬럼들: ['substrate', 'organism', 'pH', 'temperature', 'type', 'sequence', 'turnover_number', 'ec_number', 'uniprot', 'mutation', 'isomeric_smiles', 'canonical_smiles']


In [81]:
len(df_aggregated)

14718

In [83]:
df_aggregated.isna().sum()

substrate              0
organism               0
pH                     0
temperature            0
type                   0
sequence               0
turnover_number        7
ec_number              0
uniprot                0
mutation            8647
isomeric_smiles        0
canonical_smiles       0
dtype: int64

In [84]:
df_aggregated = df_aggregated.dropna(subset=['turnover_number'])

In [85]:
df_aggregated.to_csv('data/combined_brenda_sabiork_geometric_mean.csv', index=False)

In [14]:
df = pd.read_csv('data/combined_brenda_sabiork.csv')
len(df)

14730